# NOC Codes

This portion of code is meant to define and describe design decisions relating to the NOC code dimension table, as well as its subdimensions.

To begin with, we'll need to define the root of the project, as scattered `ipynb` files create something of an incoherent workspace. As this project is an application, jupyter functionally operates both as a way for me to define the project, document it, and play with the data live.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_root(markers=(".git", "requirements.txt")):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in markers):
            return p
    raise FileNotFoundError("Project root not found.")

ROOT = find_root()

As such, each call to data needs to be built from this file global.

## Object Definition

I essentially want to create the following model for noc information.

```mermaid
erDiagram
    NocDim }o--|| NocGroups : ""


    NocGroups {
        int nocgroup_id PK
        string name
    }
    NocDim {
        int level
        int nocgroup_id FK
    }
    
```

In [2]:
class Noc:
    """
    Represents NOC in a snowflake schema. Transforms the csv into the
    dimension and its outrigger tables before loading.
    """
    def __init__(self, file: str):
        self.level_ot: set[int] = set()
        self._raw_data = pd.read_csv(file)
        self._levels = self._parse_levels()

    def _parse_levels(self):
        """Parses the file """
        levels = pd.DataFrame(
            {
                "Groups": self._raw_data["Hierarchical structure"].unique()
            }
        )

        return levels

We want to see headers and then we can dive into data in different columns.

In [3]:
noc = Noc(ROOT / "data" / "noc_2021_version_1.0_-_classification_structure.csv")

In [4]:
pd.DataFrame(
    {
        "Headers": noc._raw_data.columns
    }
)

,Headers
0,Level
1,Hierarchical structure
2,Code - NOC 2021 V1.0
3,Class title
4,Class definition


Break it down column by column. Starting with level:

In [5]:
pd.DataFrame(
    {
        "Groups": noc._raw_data["Hierarchical structure"].unique()
    }
)

,Groups
0,Broad Category
1,Major Group
2,Sub-major Group
3,Minor Group
4,Unit Group
